# Metrics Demo

In [1]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

In [2]:
from crest import DataServer

lstm_streamflow = DataServer.load('lstm_streamflow')

In [3]:
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np

In [4]:
prcp_data = (lstm_streamflow['PRCP(mm/day)_maurer'].dropna(dim='date').to_numpy())

print(prcp_data)

feature_len = (prcp_data.shape)[0]
time_series_len = (prcp_data.shape)[1]

INP = 'precipitation'
OUT = 'streamflow'

print(feature_len, time_series_len)

[[ 0.          0.          0.         ... 39.56000137 10.57999992
   0.64999998]]
1 10501


In [5]:
def xy_rolling_window(data, window):
    ''' Generate a rolling window of input and target sets for the given data '''
    
    def rolling_window(data, size):
        window = lambda i: slice(i, i+size)
        result = [data[window(i)] for i in range(len(data) - size)]
        return np.stack(result, axis=0)
    
    x = rolling_window(data, window)
    y = data[window:]
    return x.reshape(len(x), window, -1), y

In [6]:
n_train = int(0.5*time_series_len)
n_test = n_train + int(0.3*time_series_len)
print(n_train, n_test)

train= prcp_data[0][:n_train]
test= prcp_data[0][n_train:n_test]
valid = prcp_data[0][n_test:]

n_seq = 60
xy_train = xy_rolling_window(train,n_seq)
xy_valid = xy_rolling_window(valid,n_seq)
xy_test = xy_rolling_window(test,n_seq)

5250 8400


In [7]:
from crest.data.loading.StructuredDataset import StructuredDataset

ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_valid = StructuredDataset(*xy_valid, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

In [8]:
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)
batch_test  = Batcher(ds_test ,  **batch_kwargs)

In [9]:
from tensorflow.keras.layers import LSTM, Dense, Input
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    #set inputs/outputs
    graph.inputs = {INP:None}
    graph.outputs = {OUT:None}
    return graph

layers = [crest_layer(layer) for layer in [
    LSTM(5, return_state=False, return_sequences=False)
]]

# Rename outputs to use as inputs (except the last layer)
for i in layers[:-1]:
    i.rename_io(outputs_map={OUT:INP})
    
# Create the CREST graph: ensure tensor shapes are specified for the inputs and outputs
graph = HTG(
    name='streamflow',
    inputs={INP : tf.TensorSpec(shape=[None, n_seq, feature_len])},
    outputs={OUT : tf.TensorSpec(shape=[None, n_seq, 5])}
)
print(list(map(graph.add_edge, ['input']+layers, layers+['output'])))

[None, None]


In [10]:
import tensorflow as tf

def reduce_mean(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

In [11]:
from crest.model.Model import Model

model = Model(graph)
model.metric.register('reduce_mean', reduce_mean)

In [12]:
build_kwargs= {
    'optimizer' : 'Adam', 
    'loss'      : 'mse', 
    'metrics'   : ['mse', 'reduce_mean', 'nse']
}

model.compile(**build_kwargs)
model.fit(batch_train, **{
    'validation_data'  : batch_valid,
    'steps_per_epoch'  : 50, 
    'validation_steps' : 5,  
    'epochs'           : 5,
    'verbose'          : 1
})

Epoch 1/5


 1/50 [..............................] - ETA: 1:24 - loss: 50.5766 - mean_squared_error: 50.5766

 3/50 [>.............................] - ETA: 1s - loss: 47.4807 - mean_squared_error: 47.4807  

 6/50 [==>...........................] - ETA: 0s - loss: 38.4016 - mean_squared_error: 38.4016

 9/50 [====>.........................] - ETA: 0s - loss: 41.1708 - mean_squared_error: 41.1708

...

38/50 [=====================>........] - ETA: 0s - loss: 38.9690 - mean_squared_error: 38.9690

41/50 [=======================>......] - ETA: 0s - loss: 38.7899 - mean_squared_error: 38.7899

45/50 [==========================>...] - ETA: 0s - loss: 39.1884 - mean_squared_error: 39.1884

49/50 [============================>.] - ETA: 0s - loss: 38.6072 - mean_squared_error: 38.6072

50/50 [==============================] - 2s 33ms/step - loss: 39.1060 - mean_squared_error: 39.1060 - val_loss: 64.3606 - val_mean_squared_error: 64.3606


In [13]:
saved_metrics = model.metric.to_json()

with open('saved_metrics.json', 'w') as f:
    f.write(saved_metrics)

In [14]:
with open('saved_metrics.json', 'r') as f:
    loaded_metrics = f.read()

# reset metrics
model.metric = model.metric.from_json(loaded_metrics)

In [15]:
batch_test  = Batcher(ds_test ,  **batch_kwargs)

with batch_test as batcher:
    res = model.evaluate(batch_test, verbose=1, return_dict=True)
    print(res)

1/1 [==============================] - ETA: 0s - loss: 34.8715 - mean_squared_error: 34.8715

1/1 [==============================] - 0s 30ms/step - loss: 34.8715 - mean_squared_error: 34.8715


{'loss': 34.871463775634766, 'mean_squared_error': 34.871463775634766}
